# Пространственная детализация OpenWeather в Петропавловске-Камчатском

Выберите источник и выполните все ячейки. По умолчанию используется **синтетический пример**. Для фактических ответов установите `SOURCE = "openweather"`. Пустой фактический CSV означает отсутствие замеров.

Основная выборка состоит из наземных точек. Группировка выполняется по продукту, раунду, этапу и точному `dt` в UTC. Даже небольшое несовпадение `dt` Current разносит строки по разным срезам. Внутри Forecast сравнивается один срок действия, окно получения всего раунда ограничено 12 минутами. Время выпуска модели ответ не сообщает. Для графиков можно выбрать конкретный срез. Контрольную точку в бухте можно рассмотреть на HTML-карте.

In [ ]:
from pathlib import Path
import sys
from collections import Counter
from statistics import median

WORKSPACE = Path.cwd()
if WORKSPACE.name == "weather_coverage":
    WORKSPACE = WORKSPACE.parent
sys.path.insert(0, str(WORKSPACE))
from weather_coverage.experiment import FIELDS, HERE, coordinates, read_csv, sampling_points
from weather_coverage.analysis import (
    cross_product, distance_summary, exact_groups, grouped, nearest_pairs, pair_data,
    rarefaction, response_groups, transect_summary,
)
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SOURCE = "demo"  # openweather для фактических ответов
SLICE = None    # по умолчанию самый полный срез Forecast, затем Current
SHOW_PRODUCT = "forecast"
SHOW_STAGE = "pilot"
path = HERE / ("observations_demo.csv" if SOURCE == "demo" else "observations.csv")
rows = read_csv(path)
land = [row for row in rows if row["on_land"] == "True"]
groups = grouped(land)
options = sorted(groups, key=lambda k: (k[1] != SHOW_PRODUCT, k[3] != SHOW_STAGE, -len(groups[k]), int(k[-1]), k))
print("СИНТЕТИЧЕСКИЕ ДАННЫЕ" if SOURCE == "demo" else "ФАКТИЧЕСКИЕ ОТВЕТЫ OPENWEATHER")
print("Строк:", len(rows), "| наземных строк:", len(land), "| точных срезов:", len(groups))
catalog = [{"срез": i, "продукт": k[1], "раунд": k[2], "этап": k[3],
            "UTC": groups[k][0]["valid_utc"], "точек": len(groups[k])}
           for i, k in enumerate(options)]
if catalog:
    catalog_table = pd.DataFrame(catalog)
    display(catalog_table.groupby(["продукт", "этап"], sort=False).head(5))
chosen = options[SLICE if SLICE is not None else 0] if options else None
sample = groups.get(chosen, [])
if chosen:
    print("Срез графиков:", catalog[options.index(chosen)])
else:
    print("Нет срезов для расчёта.")

## Уникальные значения, совпадения и ближайшие пары

Подсчёт уникальных значений использует опубликованную точность. Совпадения могут быть следствием округления, одинаковых условий или работы алгоритма провайдера. «Совпавший профиль» означает равенство всего доступного численного вектора и одинаковый набор пропусков. Расстояния рассчитаны в UTM 57N, разность направления ветра берётся по короткой дуге. Пропуски исключаются по каждому полю отдельно. Суммарная интенсивность вычисляется только при двух явно заданных объёмах, интенсивность дождя и снега можно проверить по отдельности.

In [ ]:
details = {}
summary = []
for field, label in FIELDS.items():
    valid, pairs = pair_data(sample, field)
    same = {v: ids for v, ids in exact_groups(valid).items() if len(ids) >= 2}
    nearest = nearest_pairs(pairs)
    details[field] = (valid, pairs, same, nearest)
    summary.append({"параметр": label, "доступно": len(valid), "пропусков": len(sample)-len(valid),
                    "уникальных": len(set(valid.values())), "групп совпадений": len(same),
                    "ближайших пар": len(nearest),
                    "медиана Δ ближайших": median(p["difference"] for p in nearest) if nearest else None})
display(pd.DataFrame(summary))
for field in ("temp_c", "wind_ms", "snow_mm_h"):
    print(FIELDS[field], "| точное значение → ID:", details[field][2])
print("Группы совпавших профилей:", response_groups(sample, FIELDS))
district_fields = ["point_id", "district", "temp_c", "wind_ms", "snow_mm", "visibility_m", "pop"]
district_summary = [{field: row[field] for field in district_fields} for row in sample if row["district"]]
if district_summary:
    display(pd.DataFrame(district_summary))

## Разность в зависимости от расстояния и объёма выборки

Каждая точка на диаграмме соответствует паре одного среза. Парные разности зависимы, поскольку координаты повторяются между парами. Кривая справа внизу показывает ожидаемое число уникальных значений при равновероятной выборке без возвращения из точек с заполненным полем.

In [ ]:
if sample:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout="constrained")
    for ax, fields, label in zip(axes.flat, [("temp_c",), ("wind_ms",),
                                            ("precip_mm_h", "rain_mm_h", "snow_mm_h")],
                                  ["Δ температуры, °C", "Δ скорости ветра, м/с",
                                   "Δ интенсивности осадков, мм API/ч"]):
        for field in fields:
            pairs = details[field][1]
            if pairs:
                ax.scatter([p["distance_m"]/1000 for p in pairs], [p["difference"] for p in pairs],
                           s=14, alpha=0.55, label={"precip_mm_h":"Сумма","rain_mm_h":"Дождь",
                                                     "snow_mm_h":"Снег"}.get(field))
        ax.set(xlabel="Расстояние, км", ylabel=label)
        if len(fields) > 1 and ax.has_data():
            ax.legend()
        if not ax.has_data():
            ax.text(.5,.5,"Нет сопоставимых пар",ha="center",transform=ax.transAxes)
    ax = axes[1,1]
    for field in ("temp_c", "wind_ms", "snow_mm_h"):
        curve = rarefaction(details[field][0])
        if curve:
            ax.plot(range(1, len(curve)+1), curve, marker=".", label=FIELDS[field])
    ax.set(xlabel="Число точек в подвыборке", ylabel="Ожидаемое число уникальных значений")
    if ax.has_data():
        ax.legend(fontsize=8)
    for ax in axes.flat:
        ax.grid(alpha=.2)
    fig.suptitle(("ДЕМОНСТРАЦИЯ" if SOURCE == "demo" else "OPENWEATHER")
                  + " · " + sample[0]["endpoint"] + " · " + sample[0]["valid_utc"])
    plt.show()
    bins = [{"параметр": label, "верхняя граница, м": limit, "число пар": count,
             "ненулевых": nonzero, "медиана Δ": middle}
            for field, label in FIELDS.items()
            for limit, count, nonzero, middle in distance_summary(details[field][1])]
    display(pd.DataFrame(bins))

## Проверочные пары и сравнение продуктов

Порог практической значимости задаётся отдельно по параметру в `analysis.py`. Для температуры используется 0,5 °C, для скорости ветра 1 м/с, для интенсивности 0,2 мм/ч. Пара считается кандидатом на устойчивое различие после двух разных сроков действия, прошедших порог. Несколько сроков одного прогона могут быть зависимыми, поэтому для проектного вывода требуются разные раунды и погодные эпизоды. Операционная устойчивость предполагает воспроизведение на обеих парах данного масштаба. Нижний проверенный масштаб с сигналом остаётся характеристикой выборки.

Плотный этап можно выбрать через `SHOW_STAGE = "dense"`. Сопоставление продуктов использует точный `dt` и ID точки. При наличии нескольких прогнозов берётся последний полученный до срока действия, сохраняется фактическая заблаговременность относительно запроса. Объёмы за 1 и 3 часа исключены из межпродуктового сравнения. Среднечасовые интенсивности всё равно описывают разные окна. Совпавший срок для Current и Forecast может отсутствовать.

In [ ]:
points = sampling_points(coordinates())
checks = transect_summary(groups, points)
focus = [v for v in checks if v["field"] in ("temp_c", "wind_ms", "snow_mm_h")]
if focus:
    display(pd.DataFrame(focus))
persistent = Counter((r["source"], r["endpoint"], r["field"], r["radius_m"]) for r in checks
                     if r["repeat_candidate"])
scale = {}
for (source, endpoint, field, radius), count in sorted(persistent.items()):
    if count >= 2:
        scale.setdefault((source, endpoint, field), radius)
print("Наименьший проверенный повторяемый масштаб на обоих профилях:", scale)
comparison = list(cross_product(rows, [f for f in FIELDS if f not in {"rain_mm", "snow_mm"}]))
if comparison:
    display(pd.DataFrame(comparison, columns=["источник", "точка", "UTC", "поле",
                                            "заблаговременность, ч", "Δ"]).head(30))
else:
    print("Между продуктами нет совпавших сроков с заполненными полями.")
if SOURCE == "openweather" and not any(r["snow_mm"] not in (None, "") and float(r["snow_mm"]) > 0
                                      for r in land):
    print("Пространственные различия снегопада не проверены на ненулевом снеге.")